# ЛР 02.1 — Глобальная интерпретация моделей (TODO)

## Цель
- взять лучшие неполные набор признаков (feature set) из ЛР 01;
- сравнить глобальные объяснения `LogisticRegression` и `RandomForestClassifier`;
- проверить, как native importance соотносится с `permutation importance`;
- собрать компактную summary по `partial dependence` для числовых признаков.

## Что важно
- интерпретируем не raw-модель, а модель после препроцессинга;
- не путайте importance и причинность;
- фиксируйте не только код, но и смысл различий между объяснениями.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Загрузка данных и набор признаков (feature set) из ЛР 01

Выбираем лучший **неполный** набор признаков по артефактам первой лабораторной,
чтобы интерпретация не дублировала baseline `full`.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

prepared = {}
rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    feature_set_name = lab.choose_best_nonfull_feature_set(model_results, feature_sets, dataset_name)
    selected_features = feature_sets[dataset_name][feature_set_name]
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_set_name': feature_set_name,
        'selected_features': selected_features,
    }
    rows.append({
        'dataset': dataset_name,
        'feature_set': feature_set_name,
        'n_selected_features': len(selected_features),
        'selected_preview': ', '.join(selected_features[:5]),
    })

selection_summary = pd.DataFrame(rows)
selection_summary

,dataset,feature_set,n_selected_features,selected_preview
0,medical,set_A_wrapper,10,"num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, cat__smoking_status_never"
1,finance,set_A_wrapper,10,"num__annual_income, num__credit_score, num__loan_to_income, num__delinquency_count, cat__previous_default_no"


## Шаг 2. Обучение двух моделей на выбранных набор признаков (feature set)

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [3]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

artifacts = {}
quality_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    artifacts[dataset_name] = {
        'LogisticRegression': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        ),
        'RandomForest': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=3,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    }

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in artifacts[dataset_name].items():
        metrics = lab.summarize_model_quality(artifact)
        quality_rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'feature_set': ctx['feature_set_name'],
            **metrics,
        })

quality_summary = pd.DataFrame(quality_rows).sort_values(['dataset', 'roc_auc'], ascending=[True, False])
quality_summary

c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,dataset,model,feature_set,accuracy,f1,roc_auc
2,finance,LogisticRegression,set_A_wrapper,0.700000,0.602410,0.719866
3,finance,RandomForest,set_A_wrapper,0.659091,0.468085,0.648197
0,medical,LogisticRegression,set_A_wrapper,0.694444,0.504505,0.769049
1,medical,RandomForest,set_A_wrapper,0.766667,0.343750,0.709220


## Шаг 3. Глобальные объяснения

Собираем единый long-format `global_importance_comparison`:
- `coef_abs` для линейной модели;
- `feature_importance` для дерева;
- `permutation` для обеих моделей.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [4]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

importance_frames = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        importance_frames.append(
            lab.build_global_importance_table(
                artifact=artifact,
                dataset_name=dataset_name,
                model_name=model_name,
                feature_set_name=feature_set_name,
            )
        )

global_importance_comparison = pd.concat(importance_frames, ignore_index=True)
top_global = (
    global_importance_comparison
    .sort_values(['dataset', 'model', 'method', 'rank'])
    .groupby(['dataset', 'model', 'method'], group_keys=False)
    .head(8)
)
top_global

,dataset,model,feature_set,method,feature,score,rank
40,finance,LogisticRegression,set_A_wrapper,coef_abs,num__loan_to_income,0.363093,1
41,finance,LogisticRegression,set_A_wrapper,coef_abs,num__credit_score,0.356533,2
42,finance,LogisticRegression,set_A_wrapper,coef_abs,num__annual_income,0.289840,3
43,finance,LogisticRegression,set_A_wrapper,coef_abs,cat__previous_default_no,0.253925,4
44,finance,LogisticRegression,set_A_wrapper,coef_abs,cat__previous_default_yes,0.244957,5
45,finance,LogisticRegression,set_A_wrapper,coef_abs,num__delinquency_count,0.228326,6
46,finance,LogisticRegression,set_A_wrapper,coef_abs,cat__housing_status_rent,0.203513,7
47,finance,LogisticRegression,set_A_wrapper,coef_abs,num__utilization_ratio,0.202760,8
50,finance,LogisticRegression,set_A_wrapper,permutation,num__delinquency_count,0.034917,1
51,finance,LogisticRegression,set_A_wrapper,permutation,num__credit_score,0.025705,2


## Шаг 4. Partial Dependence для числовых raw-признаков

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [5]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

curve_frames = []
summary_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        curve_df, summary_df = lab.build_partial_dependence_summary(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=feature_set_name,
        )
        curve_frames.append(curve_df)
        summary_frames.append(summary_df)

partial_dependence_curves = pd.concat(curve_frames, ignore_index=True)
partial_dependence_summary = (
    pd.concat(summary_frames, ignore_index=True)
    .sort_values(['dataset', 'score_delta'], ascending=[True, False])
    .reset_index(drop=True)
)
partial_dependence_summary

,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
0,finance,LogisticRegression,set_A_wrapper,credit_score,568.35000,761.92000,0.368870,0.570613,0.201743,non_increasing
1,finance,RandomForest,set_A_wrapper,loan_to_income,0.10373,0.80536,0.332876,0.533151,0.200275,non_monotonic
2,finance,RandomForest,set_A_wrapper,annual_income,31534.01500,94086.29000,0.304620,0.499389,0.194769,non_monotonic
3,finance,RandomForest,set_A_wrapper,credit_score,568.35000,761.92000,0.316831,0.507869,0.191038,non_monotonic
4,finance,LogisticRegression,set_A_wrapper,loan_to_income,0.10373,0.80536,0.390303,0.572309,0.182006,non_decreasing
5,finance,LogisticRegression,set_A_wrapper,annual_income,31534.01500,94086.29000,0.385582,0.558018,0.172436,non_increasing
6,medical,LogisticRegression,set_A_wrapper,age,34.00000,76.00000,0.198070,0.637378,0.439309,non_decreasing
7,medical,LogisticRegression,set_A_wrapper,cholesterol,158.98000,251.67000,0.284773,0.588334,0.303561,non_decreasing
8,medical,LogisticRegression,set_A_wrapper,systolic_bp,105.18000,151.90000,0.281265,0.549319,0.268054,non_decreasing
9,medical,RandomForest,set_A_wrapper,age,34.00000,76.00000,0.149874,0.408999,0.259125,non_monotonic


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о различиях между native importance и permutation importance.
- Native importance (коэффициенты логистической регрессии и важности из RandomForest) показывают, какие признаки модель считает важными внутри своей структуры, но не учитывают корреляции между признаками. Permutation importance, в отличие от них, измеряет реальное падение качества при перемешивании признака — это более робастная оценка, но она дороже по вычислениям и зависит от метрики.
- Для логистической регрессии коэффициенты и permutation importance хорошо согласуются, так как модель линейна. Для RandomForest native importance может завышать важность признаков с большим количеством уникальных значений (например, числовых), тогда как permutation важность корректирует эту оценку.
- TODO (обязательно): поясните, почему partial dependence считается модельно-зависимым упрощением.
- Partial dependence показывает, как изменяется предсказание модели при изменении одного признака в среднем по всем остальным. Это модельно-зависимое упрощение, потому что оно строится на основе конкретной обученной модели и не учитывает взаимодействия признаков в полной мере. На практике оно помогает понять направление влияния признака (монотонное, пороговое, немонотонное).

### Сравнение с альтернативами
- TODO (обязательно): сравните минимум два способа глобальной интерпретации.
- Формат: когда один подход полезнее другого и почему.
- Native importance быстрее вычисляется и встроена в модель, но может быть смещена. Permutation importance даёт более честную оценку вклада признака, но требует пересчёта метрики на тестовой выборке многократно. Native importance подходит для быстрой первичной оценки, permutation — для валидации и сравнения моделей.
- Partial dependence хорошо работает для числовых признаков и показывает усреднённый эффект. Альтернативой являются SHAP и LIME, которые дают локальные объяснения, но глобально агрегируются сложнее. Partial dependence полезна, когда нужно понять общий тренд, а не отдельные выбросы.

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).
- [scikit-learn: Permutation Importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [scikit-learn: Partial Dependence Plots](https://scikit-learn.org/stable/modules/partial_dependence.html)
- `study-notes/glossary.md` — термины `native importance`, `permutation importance`, `partial dependence`.

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.
`Глоссарий обновлен: native importance, permutation importance, partial dependence`

## Контрольные точки
1. В `global_importance_comparison` есть колонки `dataset, model, feature_set, method, feature, score, rank`.
2. Для каждого датасета есть обе модели.
3. В `partial_dependence_summary` есть минимум по 2-3 raw-признака на датасет.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Согласованность глобальных объяснений
- Возьмите `global_importance_comparison`.
- Для каждой тройки `dataset + model + method` проверьте top-5 и top-8 признаков.
- Коротко опишите, какие признаки стабильно держатся в топе, а какие зависят от метода.

### Задание 2. Интерпретация partial dependence
- Возьмите `partial_dependence_summary`.
- Найдите признаки с максимальным `score_delta`.
- Поясните, где тренд выглядит монотонным, а где нет.

### Задание 3. Экспорт артефактов и краткий вывод
- Сохраните `outputs/global_importance_comparison.csv`.
- Сохраните `outputs/partial_dependence_summary.csv`.
- Добавьте 2-3 коротких вывода отдельно для `medical` и `finance`.

Выводы:</br>
Medical: в обеих моделях стабильно важны возраст, систолическое давление, холестерин и физическая активность. RandomForest выделяет глюкозу, логистическая регрессия — ИМТ.</br>
Finance: общие топы — loan_to_income, annual_income, credit_score. Permutation важность подчёркивает delinquency_count, native важность RandomForest — savings_balance.</br>
Partial dependence: score_delta выше у признаков с немонотонным или пороговым эффектом (например, возраст, credit_score).</br>

In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для global_importance_comparison и partial_dependence_summary.
# 2) Сохраните оба DataFrame в CSV внутри outputs/.
# 3) Добавьте текстовую интерпретацию в markdown-блоки выше.

# Переименовываем колонку raw_feature в feature для partial_dependence_summary
partial_dependence_summary = partial_dependence_summary.rename(columns={'raw_feature': 'feature'})

required_columns_importance = {'dataset', 'model', 'feature_set', 'method', 'feature', 'score', 'rank'}
required_columns_pd = {'dataset', 'model', 'feature_set', 'feature', 'score_delta', 'trend_type'}

# Проверяем наличие колонок и выводим информацию о недостающих
missing_importance = required_columns_importance - set(global_importance_comparison.columns)
missing_pd = required_columns_pd - set(partial_dependence_summary.columns)

if missing_importance:
    print(f"Missing columns in global_importance_comparison: {missing_importance}")
    print(f"Available columns: {global_importance_comparison.columns.tolist()}")
    
if missing_pd:
    print(f"Missing columns in partial_dependence_summary: {missing_pd}")
    print(f"Available columns: {partial_dependence_summary.columns.tolist()}")

assert not missing_importance, f"Missing columns in global_importance_comparison: {missing_importance}"
assert not missing_pd, f"Missing columns in partial_dependence_summary: {missing_pd}"

# Сохраняем DataFrame
global_importance_comparison.to_csv(OUTPUT_DIR / 'global_importance_comparison.csv', index=False)
partial_dependence_summary.to_csv(OUTPUT_DIR / 'partial_dependence_summary.csv', index=False)

print("Saved: outputs/global_importance_comparison.csv")
print("Saved: outputs/partial_dependence_summary.csv")

print("\n=== Global Importance: Medical ===")
print(global_importance_comparison[global_importance_comparison['dataset'] == 'medical'].groupby(['model', 'method']).head(5))

print("\n=== Global Importance: Finance ===")
print(global_importance_comparison[global_importance_comparison['dataset'] == 'finance'].groupby(['model', 'method']).head(5))

print("\n=== Partial Dependence Summary: Medical ===")
print(partial_dependence_summary[partial_dependence_summary['dataset'] == 'medical'])

print("\n=== Partial Dependence Summary: Finance ===")
print(partial_dependence_summary[partial_dependence_summary['dataset'] == 'finance'])


Saved: outputs/global_importance_comparison.csv
Saved: outputs/partial_dependence_summary.csv

=== Global Importance: Medical ===
    dataset               model    feature_set              method  \
0   medical  LogisticRegression  set_A_wrapper            coef_abs   
1   medical  LogisticRegression  set_A_wrapper            coef_abs   
2   medical  LogisticRegression  set_A_wrapper            coef_abs   
3   medical  LogisticRegression  set_A_wrapper            coef_abs   
4   medical  LogisticRegression  set_A_wrapper            coef_abs   
10  medical  LogisticRegression  set_A_wrapper         permutation   
11  medical  LogisticRegression  set_A_wrapper         permutation   
12  medical  LogisticRegression  set_A_wrapper         permutation   
13  medical  LogisticRegression  set_A_wrapper         permutation   
14  medical  LogisticRegression  set_A_wrapper         permutation   
20  medical        RandomForest  set_A_wrapper  feature_importance   
21  medical        RandomFores